# Empirical bias-variance decomposition for polynomial regression

This notebook matches notes.md §4 (the bias-variance decomposition) and §5 (k-fold
cross-validation), using numpy only.

The true function is $f(x) = \sin(2\pi x)$ on $[0, 1]$, and observations are
$y = f(x) + \varepsilon$ with $\varepsilon \sim N(0, 0.3^2)$, so the irreducible error is
$\sigma^2 = 0.09$. For each polynomial degree from 1 to 15, the notebook fits 500
independently drawn training sets and measures, on a fixed grid of test points:

```text
bias^2   = mean_x ( f(x) - avg_D fhat_D(x) )^2
variance = mean_x avg_D ( fhat_D(x) - avg_D fhat_D(x) )^2
total expected test error = bias^2 + variance + sigma^2
```

It checks that total against a direct measurement on fresh noisy test labels, then runs
5-fold cross-validation on one training set to show that CV picks a degree close to the
one with the lowest true error.

Run: `uv run jupyter execute ml/05_evaluation/bias_variance_experiment.ipynb`

## Setup: constants and the polynomial fit

`fit_poly` solves the least-squares normal equations on a Vandermonde matrix (columns
$1, x, x^2, \dots$), with a ridge term of $10^{-8}$ added to the diagonal so the solve stays
numerically stable at degree 15. `predict_poly` evaluates the fitted polynomial.

In [1]:
import numpy as np

np.random.seed(42)

SIGMA = 0.3          # noise std -> irreducible error sigma^2 = 0.09
N_TRAIN = 30         # points per training set
N_SETS = 500         # number of resampled training sets
DEGREES = range(1, 16)


def f_true(x):
    return np.sin(2 * np.pi * x)


def fit_poly(x, y, degree, ridge=1e-8):
    """Least-squares polynomial fit via the normal equations on a Vandermonde
    matrix (tiny ridge for numerical stability at high degree). Returns coeffs."""
    X = np.vander(x, degree + 1, increasing=True)
    A = X.T @ X + ridge * np.eye(degree + 1)
    return np.linalg.solve(A, X.T @ y)


def predict_poly(coeffs, x):
    X = np.vander(x, len(coeffs), increasing=True)
    return X @ coeffs

## 1. Bias², variance and noise for degrees 1 to 15

The cell draws 500 training sets of 30 points each, the same sets for every degree. For
each degree it fits all 500, averages their predictions on 200 test points to get
$\bar{f}$, and computes bias² (squared gap between $\bar{f}$ and $f$) and variance
(spread of the 500 fits around $\bar{f}$). The last column draws fresh noisy labels
$y = f + \varepsilon$ at the test points and measures the squared error directly.

Expect bias² to drop to near 0 by degree 3 and variance to grow quickly with degree
after that, so their sum plus 0.09 is U-shaped, and expect the last two columns to agree
up to sampling noise. Two details in the table: degree 2 has about the same bias as degree 1
because $\sin(2\pi x)$ is antisymmetric about $x = 0.5$, so a quadratic term does not
help on average; and the small rise in bias² at high degree is mostly an estimation
effect, because $\bar{f}$ is an average of only 500 fits, so the measured bias² includes
about variance/500 (for degree 15, $9.3/500 \approx 0.019$).

In [2]:
def bias_variance_table():
    rng = np.random.default_rng(42)
    x_test = np.linspace(0.0, 1.0, 200)
    f_test = f_true(x_test)

    # Pre-draw all training sets so every degree sees the same data draws.
    xs = [rng.uniform(0, 1, N_TRAIN) for _ in range(N_SETS)]
    ys = [f_true(x) + rng.normal(0, SIGMA, N_TRAIN) for x in xs]

    print(f"Bias^2 / variance decomposition: {N_SETS} training sets of n={N_TRAIN}, "
          f"sigma^2={SIGMA**2:.3f}")
    header = (f"{'degree':>6} {'bias^2':>10} {'variance':>10} {'noise':>8} "
              f"{'b2+var+noise':>13} {'measured err':>13}")
    print(header)
    print("-" * len(header))

    results = []
    for d in DEGREES:
        preds = np.empty((N_SETS, x_test.size))
        for i in range(N_SETS):
            preds[i] = predict_poly(fit_poly(xs[i], ys[i], d), x_test)

        avg_pred = preds.mean(axis=0)
        bias2 = float(np.mean((f_test - avg_pred) ** 2))
        variance = float(np.mean(preds.var(axis=0)))
        decomposed = bias2 + variance + SIGMA ** 2

        # Directly measured expected test error: draw fresh noisy test labels
        # y = f + noise for every training set and average (y - fhat)^2.
        y_test = f_test + rng.normal(0, SIGMA, preds.shape)
        measured = float(np.mean((y_test - preds) ** 2))

        results.append((d, bias2, variance, decomposed))
        print(f"{d:>6} {bias2:>10.5f} {variance:>10.5f} {SIGMA**2:>8.3f} "
              f"{decomposed:>13.5f} {measured:>13.5f}")

    best = min(results, key=lambda r: r[3])
    print("-" * len(header))
    print(f"Bias^2 drops to near 0 by degree 3 and variance grows with degree after that; "
          f"total error is U-shaped, minimized at degree {best[0]} ({best[3]:.5f}).")
    print("Sanity check: bias^2 + variance + sigma^2 matches the directly "
          "measured expected error (last two columns agree).")
    return best[0]


best_decomp = bias_variance_table()

Bias^2 / variance decomposition: 500 training sets of n=30, sigma^2=0.090
degree     bias^2   variance    noise  b2+var+noise  measured err
-----------------------------------------------------------------
     1    0.20038    0.02281    0.090       0.31319       0.31326
     2    0.20274    0.05437    0.090       0.34711       0.34666
     3    0.00481    0.01810    0.090       0.11291       0.11255
     4    0.00528    0.03325    0.090       0.12854       0.12840
     5    0.00013    0.05329    0.090       0.14342       0.14347
     6    0.00014    0.07929    0.090       0.16943       0.16919
     7    0.00029    0.13794    0.090       0.22823       0.22703
     8    0.00051    0.21425    0.090       0.30475       0.30546
     9    0.00070    0.35853    0.090       0.44923       0.44743
    10    0.00097    0.63963    0.090       0.73061       0.72794
    11    0.00153    1.13357    0.090       1.22510       1.22247
    12    0.00262    1.95500    0.090       2.04762       2.05129


    13    0.00467    3.30085    0.090       3.39552       3.39738
    14    0.00848    5.53132    0.090       5.62980       5.62493
    15    0.01552    9.30229    0.090       9.40781       9.40383
-----------------------------------------------------------------
Bias^2 drops to near 0 by degree 3 and variance grows with degree after that; total error is U-shaped, minimized at degree 3 (0.11291).
Sanity check: bias^2 + variance + sigma^2 matches the directly measured expected error (last two columns agree).


## 2. k-fold cross-validation on a single training set

In practice there is one training set, not 500, so bias² and variance cannot be measured
directly. The cell draws one set of 60 points, splits it into 5 folds, and for each
degree trains on 4 folds and computes MSE on the held-out fold, 5 times. It also reports
the training MSE of a fit on all 60 points.

Expect training MSE to keep falling with degree (apart from tiny numerical blips), while
CV MSE is lowest at a low degree and larger and more variable across folds at high
degree. With only 60 points, the CV minimum can sit well below $\sigma^2 = 0.09$ when
the sampled noise happens to be small, so the cell also prints the mean squared noise
of this particular sample.

In [3]:
def kfold_indices(n, k, rng):
    idx = rng.permutation(n)
    return np.array_split(idx, k)


def kfold_cv_demo():
    print("k-fold CV on a single training set of n=60 (k=5)")
    rng = np.random.default_rng(42)
    n, k = 60, 5
    x = rng.uniform(0, 1, n)
    y = f_true(x) + rng.normal(0, SIGMA, n)
    folds = kfold_indices(n, k, rng)
    noise_ms = float(np.mean((y - f_true(x)) ** 2))

    header = f"{'degree':>6} {'CV MSE (mean)':>14} {'CV MSE (std)':>13} {'train MSE':>10}"
    print(header)
    print("-" * len(header))

    cv_results = []
    for d in DEGREES:
        fold_mses = []
        for i in range(k):
            val = folds[i]
            trn = np.concatenate([folds[j] for j in range(k) if j != i])
            coeffs = fit_poly(x[trn], y[trn], d)
            fold_mses.append(np.mean((y[val] - predict_poly(coeffs, x[val])) ** 2))
        coeffs_full = fit_poly(x, y, d)
        train_mse = float(np.mean((y - predict_poly(coeffs_full, x)) ** 2))
        m, s = float(np.mean(fold_mses)), float(np.std(fold_mses))
        cv_results.append((d, m, s))
        print(f"{d:>6} {m:>14.5f} {s:>13.5f} {train_mse:>10.5f}")

    best_d, best_m, _ = min(cv_results, key=lambda r: r[1])
    print("-" * len(header))
    print(f"CV selects degree {best_d}.")
    print("Train MSE falls with degree (exact least squares on nested models can never")
    print("increase it; the tiny ridge and floating-point error cause small blips at high degree),")
    print("while CV MSE is lowest at degree 3 and larger and more variable at higher degrees.")
    print(f"The CV minimum ({best_m:.3f}) is below sigma^2 = {SIGMA**2:.3f} because the noise drawn")
    print(f"for this sample is small: its mean square is {noise_ms:.3f}.")
    print("CV MSE far below sigma^2 on a large validation set would signal a bug or leakage.")
    return best_d


best_cv = kfold_cv_demo()

k-fold CV on a single training set of n=60 (k=5)
degree  CV MSE (mean)  CV MSE (std)  train MSE
----------------------------------------------
     1        0.30640       0.17477    0.26767
     2        0.34999       0.23642    0.26739
     3        0.05262       0.01424    0.04858
     4        0.07192       0.04650    0.04856
     5        0.07000       0.04392    0.04730
     6        0.10974       0.12461    0.04643
     7        0.10341       0.11850    0.04252
     8        0.10040       0.10847    0.04141
     9        0.11409       0.13223    0.04157
    10        0.11690       0.13911    0.04088
    11        0.10972       0.12767    0.03988
    12        0.10074       0.11146    0.03930
    13        0.09367       0.09752    0.03915
    14        0.08952       0.08846    0.03917
    15        0.08818       0.08469    0.03921
----------------------------------------------
CV selects degree 3.
Train MSE falls with degree (exact least squares on nested models can never
increase

## 3. Compare the two choices

The degree with the lowest true expected error (section 1, which used 500 training sets
and the known $f$) is compared with the degree chosen by 5-fold CV from one set of 60
noisy points. Expect them to be equal or close.

In [4]:
print(f"Degree with lowest true expected error: {best_decomp}; "
      f"degree chosen by 5-fold CV: {best_cv}.")
print("CV, using only one finite noisy training set, lands close to the truth.")

Degree with lowest true expected error: 3; degree chosen by 5-fold CV: 3.
CV, using only one finite noisy training set, lands close to the truth.
